In [435]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/code-mixed-hinglish-abusive-and-hate-speech/train.csv


In [436]:
df=pd.read_csv('/kaggle/input/code-mixed-hinglish-abusive-and-hate-speech/train.csv')
df

,text,label
0,Thank you so much bhai,not offensive
1,bhen ke lode,offensive
2,bh3n k3 l0d3,offensive
3,madarchod saale,offensive
4,madrch0d harami,offensive
...,...,...
1361,bhai tu toh legend hai legend,not offensive
1362,tu toh dil se dil tak jaata hai yaar,not offensive
1363,arey tu toh mera real bhai hai,not offensive
1364,tu toh hamesha yaad rahega bhai,not offensive


In [437]:
df[df.isna()==True].count()

text     0
label    0
dtype: int64

In [438]:
df.copy()
y=df['label']
y = y.map({'offensive': 1, 'not offensive': 0})
y

0       0
1       1
2       1
3       1
4       1
       ..
1361    0
1362    0
1363    0
1364    0
1365    0
Name: label, Length: 1366, dtype: int64

# Text Preprocessing

In [439]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [440]:
corpus=[]
text=df['text']
for i in text:
    corpus.append(i)

In [441]:
from sklearn.feature_extraction.text import CountVectorizer
import joblib

tf = CountVectorizer(max_features=5)
s = tf.fit_transform(corpus)
x = s.toarray()

joblib.dump(tf, "vectorizer.joblib")

['vectorizer.joblib']

In [442]:
x=pd.DataFrame(x)

**Train_test_split**

In [443]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=123)

**Model**

In [444]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score
lgbm_clf = lgb.LGBMClassifier(objective='binary', num_leaves=31, learning_rate=0.05, n_estimators=100)
lgbm_clf.fit(x_train,y_train)
lgbm_clf.booster_.save_model("model.txt")

[LightGBM] [Info] Number of positive: 485, number of negative: 471
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000107 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16
[LightGBM] [Info] Number of data points in the train set: 956, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.507322 -> initscore=0.029291
[LightGBM] [Info] Start training from score 0.029291
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li

In [445]:
y_pred = lgbm_clf.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9415


In [446]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

false_positives = cm[0, 1]
false_negatives = cm[1, 0]

print(f"\nType I Error (False Positives): {false_positives}")
print(f"\nType II Error (False Negatives): {false_negatives}")

Confusion Matrix:
 [[179  21]
 [  3 207]]

Type I Error (False Positives): 21

Type II Error (False Negatives): 3


In [447]:
model = lgb.Booster(model_file="/kaggle/working/model.txt")

tf = joblib.load("/kaggle/working/vectorizer.joblib")
sentence=input('enter ur comment :')
corpus=[]
corpus.append(sentence)
s = tf.transform(corpus)
x = s.toarray()
p=model.predict(x)
v=p[0]
if (v > 0.5):
    print(f"\n - offensive word \n - confidence={v:.4f} ")
else:
    print(f"\n - not offensive word \n - confidence={v:.4f} ")

enter ur comment : maa ki chut



 - offensive word 
 - confidence=0.8565 
